
# Explainable Pathology-Aware Relational Learning for Osteosarcoma Histopathology

### PTRN + TBEN + PCCAECN/PARCEN — Colab implementation

This notebook implements a practical, end-to-end version of the framework described in the supplied manuscript:

- Macenko-style stain normalization
- Otsu tissue masking
- Median filtering
- CLAHE
- Min-max normalization
- GLCM/LBP/Gabor/edge/morphology/distance features
- Pathology-aware patch tokens
- **PTRN**: multi-scale pathology features, pathology-difference-guided sparse neighborhoods,
  MCPRG and texture-structure relational attention
- **TBEN**: weak CAM pseudo-mask initialization, signed-distance state, reliability-aware
  boundary evolution, hard-band iterative refinement, deep supervision and inference filtering
- **PCCAECN/PARCEN**: five evidence streams, evidence projection, conflict/reliability
  estimation, reliability-gated consensus and temperature calibration
- Patient-level stratification
- Accuracy / precision / recall / F1 / MCC
- Confusion matrix, training curves, localization visualization
- Integrated-gradient-style image attribution and leave-one-stream-out evidence attribution

> Important: the manuscript reports the architecture and experimental protocol, but the result
> tables in the supplied document do not contain numeric values. This notebook therefore trains
> the model on the real dataset rather than fabricating the reported results.



## 1. Dataset described by the manuscript

The paper describes the TCIA Osteosarcoma Tumor Assessment dataset as:

- 1,144 H&E images
- 1024 × 1024 source resolution
- 3 classes: **Non-Tumor, Viable Tumor, Necrotic Tumor**
- class counts: 536 / 345 / 263
- samples originating from 50 patients
- patient-level grouping is required to avoid leakage

The manuscript's data availability points to the TCIA collection.

For Colab, this notebook supports either:
1. a local/Google Drive image directory with class folders, or
2. a CSV manifest containing image paths and labels, optionally with patient IDs.


In [ ]:

# ============================================================
# 2. INSTALLATION
# ============================================================
!pip -q install scikit-image opencv-python-headless pandas numpy matplotlib seaborn     scikit-learn pillow tqdm openpyxl scipy

import os, gc, math, json, random, warnings, copy, time
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import cv2
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore")
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from sklearn.model_selection import GroupShuffleSplit, StratifiedGroupKFold
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support, matthews_corrcoef,
    confusion_matrix, roc_auc_score, classification_report
)
from skimage.feature import graycomatrix, graycoprops, local_binary_pattern
from skimage.filters import gabor
from skimage.morphology import remove_small_objects, remove_small_holes
from skimage.measure import label, regionprops
from skimage.segmentation import find_boundaries
from skimage.transform import resize
from scipy.ndimage import distance_transform_edt

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


In [ ]:

# ============================================================
# 3. CONFIGURATION
# ============================================================
# Change only these paths/settings for your dataset.
DRIVE_ROOT = "/content/drive/MyDrive/Osteosarcoma_PTRN_TBEN"
DATA_ROOT = os.path.join(DRIVE_ROOT, "Dataset")

# Supported layouts:
# DATA_ROOT/
#   Non-Tumor/*.png
#   Viable Tumor/*.png
#   Necrotic Tumor/*.png
#
# OR:
# DATA_ROOT/manifest.csv
# with columns: image_path, label, and optional patient_id

CLASS_NAMES = ["Non-Tumor", "Viable Tumor", "Necrotic Tumor"]
CLASS_TO_ID = {c:i for i,c in enumerate(CLASS_NAMES)}

IMG_SIZE = 224
PATCH_SIZE = 16
EMBED_DIM = 128
PATH_DESC_DIM = 32
TOP_K = 8
REL_LAYERS = 2
REL_HEADS = 4

BATCH_SIZE = 8
EPOCHS = 10
LR = 2e-4
WEIGHT_DECAY = 1e-4
PATIENCE = 4
NUM_WORKERS = 2

TBEN_STEPS = 3
TBEN_STEP_SIZE = 0.35
TBEN_BAND = 0.15

USE_PRETRAINED_CONVNEXT = True
SAVE_DIR = os.path.join(DRIVE_ROOT, "Outputs")
os.makedirs(SAVE_DIR, exist_ok=True)

print("Output directory:", SAVE_DIR)


In [ ]:

# ============================================================
# 4. OPTIONAL GOOGLE DRIVE MOUNT
# ============================================================
try:
    from google.colab import drive
    drive.mount("/content/drive")
except Exception:
    print("Not running inside Colab; continuing with local paths.")


In [ ]:

# ============================================================
# 5. DATASET MANIFEST BUILDER
# ============================================================
IMG_EXTS = {".png", ".jpg", ".jpeg", ".tif", ".tiff", ".bmp"}

def infer_patient_id(path):
    # Common practical conventions: patient_001, patient-001, case001.
    stem = Path(path).stem
    parts = stem.replace("-", "_").split("_")
    for p in parts:
        q = p.lower()
        if q.startswith(("patient", "case", "subject", "specimen")):
            return p
    # Conservative fallback: use the first filename token.
    return parts[0] if parts else stem

def build_manifest(root):
    root = Path(root)
    manifest_csv = root / "manifest.csv"
    if manifest_csv.exists():
        df = pd.read_csv(manifest_csv)
        required = {"image_path", "label"}
        missing = required - set(df.columns)
        if missing:
            raise ValueError(f"manifest.csv missing columns: {missing}")
        if "patient_id" not in df.columns:
            df["patient_id"] = df["image_path"].map(infer_patient_id)
        df["image_path"] = df["image_path"].apply(
            lambda x: str((root / x).resolve()) if not os.path.isabs(str(x)) else str(x)
        )
        return df

    rows = []
    for class_name in CLASS_NAMES:
        class_dir = root / class_name
        if not class_dir.exists():
            continue
        for p in class_dir.rglob("*"):
            if p.suffix.lower() in IMG_EXTS:
                rows.append({
                    "image_path": str(p),
                    "label": class_name,
                    "patient_id": infer_patient_id(str(p))
                })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise FileNotFoundError(
            f"No images found under {root}. "
            "Create class folders or provide Dataset/manifest.csv."
        )
    df["label_id"] = df["label"].map(CLASS_TO_ID)
    return df

df = build_manifest(DATA_ROOT)
df = df.dropna(subset=["image_path", "label"]).copy()
df["label_id"] = df["label"].map(CLASS_TO_ID)
df = df[df["label_id"].notna()].copy()
df["label_id"] = df["label_id"].astype(int)

print("Images:", len(df))
print(df["label"].value_counts())
print("Patients:", df["patient_id"].nunique())
display(df.head())


In [ ]:

# ============================================================
# 6. PATIENT-LEVEL TRAIN / VALIDATION / TEST SPLIT
# ============================================================
# The manuscript explicitly requires patient-level separation.

def patient_level_split(data, test_size=0.20, val_size=0.15, seed=42):
    data = data.reset_index(drop=True)

    gss = GroupShuffleSplit(n_splits=1, test_size=test_size, random_state=seed)
    trval_idx, test_idx = next(gss.split(data, data["label_id"], groups=data["patient_id"]))
    trval = data.iloc[trval_idx].reset_index(drop=True)
    test = data.iloc[test_idx].reset_index(drop=True)

    relative_val = val_size / (1.0 - test_size)
    gss2 = GroupShuffleSplit(n_splits=1, test_size=relative_val, random_state=seed + 1)
    train_idx, val_idx = next(
        gss2.split(trval, trval["label_id"], groups=trval["patient_id"])
    )
    train = trval.iloc[train_idx].reset_index(drop=True)
    val = trval.iloc[val_idx].reset_index(drop=True)
    return train, val, test

train_df, val_df, test_df = patient_level_split(df)

print("Train:", len(train_df), "patients:", train_df.patient_id.nunique())
print("Val  :", len(val_df), "patients:", val_df.patient_id.nunique())
print("Test :", len(test_df), "patients:", test_df.patient_id.nunique())

print("\nClass distributions:")
display(pd.DataFrame({
    "Train": train_df.label.value_counts(),
    "Validation": val_df.label.value_counts(),
    "Test": test_df.label.value_counts()
}).fillna(0).astype(int))


In [ ]:

# ============================================================
# 7. PATHOLOGY PREPROCESSING
# ============================================================
def macenko_style_normalization(img_rgb, reference=None):
    # Robust Macenko-style approximation using optical density + PCA.
    # A reference image can be supplied for stricter stain-template matching.
    img = img_rgb.astype(np.float32) + 1.0
    OD = -np.log(img / 256.0)
    OD = OD.reshape(-1, 3)

    mask = np.linalg.norm(OD, axis=1) > 0.15
    if mask.sum() < 20:
        return img_rgb.copy()

    od = OD[mask]
    cov = np.cov(od, rowvar=False)
    vals, vecs = np.linalg.eigh(cov)
    order = np.argsort(vals)[::-1]
    vecs = vecs[:, order[:2]]
    proj = od @ vecs
    angles = np.arctan2(proj[:,1], proj[:,0])
    lo, hi = np.percentile(angles, [1, 99])
    v1 = vecs @ np.array([np.cos(lo), np.sin(lo)])
    v2 = vecs @ np.array([np.cos(hi), np.sin(hi)])
    stain = np.stack([v1, v2], axis=1)
    stain = stain / (np.linalg.norm(stain, axis=0, keepdims=True) + 1e-8)

    # Stable color normalization.
    target = np.array([[0.65, 0.70], [0.07, 0.15], [0.70, 0.60]], dtype=np.float32)
    target /= np.linalg.norm(target, axis=0, keepdims=True)

    try:
        C = np.linalg.lstsq(stain, od.T, rcond=None)[0]
        maxC = np.percentile(C, 99, axis=1)
        target_max = np.array([1.0, 1.0])
        Cn = C * (target_max[:,None] / (maxC[:,None] + 1e-6))
        ODn = target @ Cn
        out = np.exp(-ODn.T).reshape(img_rgb.shape)
        return np.clip(out * 255.0 - 1.0, 0, 255).astype(np.uint8)
    except Exception:
        return img_rgb.copy()

def pathology_preprocess(path, out_size=IMG_SIZE):
    img = np.array(Image.open(path).convert("RGB"))
    img = cv2.resize(img, (out_size, out_size), interpolation=cv2.INTER_AREA)

    # Step 1: Macenko-style stain normalization
    img = macenko_style_normalization(img)

    # Step 2: Otsu tissue mask
    gray = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY)
    _, tissue = cv2.threshold(gray, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    tissue = (tissue > 0).astype(np.uint8)

    # Step 3: Median filtering
    med = cv2.medianBlur(img, 3)

    # Step 4: CLAHE
    lab = cv2.cvtColor(med, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    l = clahe.apply(l)
    enhanced = cv2.cvtColor(cv2.merge([l,a,b]), cv2.COLOR_LAB2RGB)

    # Preserve tissue while retaining enough background for CNN context.
    mask3 = tissue[...,None].astype(np.float32)
    enhanced = enhanced.astype(np.float32) * (0.75 + 0.25 * mask3)

    # Step 5: Min-max normalization
    mn, mx = enhanced.min(), enhanced.max()
    enhanced = (enhanced - mn) / (mx - mn + 1e-8)

    return enhanced.astype(np.float32), tissue.astype(np.float32)

def show_preprocessing(path):
    raw = np.array(Image.open(path).convert("RGB"))
    proc, tissue = pathology_preprocess(path)
    fig, ax = plt.subplots(1,3, figsize=(15,5))
    ax[0].imshow(raw); ax[0].set_title("Original"); ax[0].axis("off")
    ax[1].imshow(proc); ax[1].set_title("Preprocessed"); ax[1].axis("off")
    ax[2].imshow(tissue, cmap="gray"); ax[2].set_title("Otsu tissue mask"); ax[2].axis("off")
    plt.tight_layout()
    plt.show()

show_preprocessing(train_df.iloc[0].image_path)


In [ ]:

# ============================================================
# 8. TEXTURE / EDGE / MORPHOLOGY DESCRIPTORS
# ============================================================
def descriptor_map(image, tissue_mask, grid=14):
    H, W = image.shape[:2]
    gray = cv2.cvtColor((image*255).astype(np.uint8), cv2.COLOR_RGB2GRAY) / 255.0

    # Gabor responses
    gabors = []
    for theta in (0, np.pi/4, np.pi/2, 3*np.pi/4):
        real, imag = gabor(gray, frequency=0.15, theta=theta)
        gabors.append(np.sqrt(real**2 + imag**2))
    gabor_map = np.mean(gabors, axis=0)

    # LBP
    lbp = local_binary_pattern((gray*255).astype(np.uint8), P=8, R=1, method="uniform")
    lbp = lbp / (lbp.max() + 1e-8)

    # Canny + morphological gradient
    edges = cv2.Canny((gray*255).astype(np.uint8), 60, 140) / 255.0
    kernel = np.ones((3,3), np.uint8)
    morph = cv2.morphologyEx((tissue_mask*255).astype(np.uint8), cv2.MORPH_GRADIENT) / 255.0

    # Distance transform
    dt = distance_transform_edt(tissue_mask > 0.5)
    dt = dt / (dt.max() + 1e-8)

    # Patch descriptor vector:
    # GLCM contrast/correlation/energy/homogeneity + LBP + Gabor +
    # edge + morphology + distance.
    cell_h, cell_w = H//grid, W//grid
    feats = []
    for r in range(grid):
        for c in range(grid):
            y0,y1 = r*cell_h, (r+1)*cell_h if r < grid-1 else H
            x0,x1 = c*cell_w, (c+1)*cell_w if c < grid-1 else W
            g = gray[y0:y1, x0:x1]
            q = np.clip((g*7).astype(np.uint8), 0, 7)
            try:
                glcm = graycomatrix(q, [1], [0], levels=8, symmetric=True, normed=True)
                gl = [
                    graycoprops(glcm, "contrast")[0,0],
                    graycoprops(glcm, "correlation")[0,0],
                    graycoprops(glcm, "energy")[0,0],
                    graycoprops(glcm, "homogeneity")[0,0],
                ]
            except Exception:
                gl = [0,0,0,0]

            feats.append([
                *gl,
                float(np.mean(lbp[y0:y1,x0:x1])),
                float(np.mean(gabor_map[y0:y1,x0:x1])),
                float(np.mean(edges[y0:y1,x0:x1])),
                float(np.mean(morph[y0:y1,x0:x1])),
                float(np.mean(dt[y0:y1,x0:x1])),
                float(np.mean(tissue_mask[y0:y1,x0:x1])),
            ])

    feat = np.asarray(feats, dtype=np.float32)
    # Robust standardization per image
    feat = (feat - np.median(feat, axis=0, keepdims=True)) / (
        np.median(np.abs(feat - np.median(feat, axis=0, keepdims=True)), axis=0, keepdims=True) + 1e-5
    )
    feat = np.nan_to_num(feat, nan=0.0, posinf=0.0, neginf=0.0)

    return feat, {
        "edge": edges.astype(np.float32),
        "morph": morph.astype(np.float32),
        "distance": dt.astype(np.float32),
        "gray": gray.astype(np.float32)
    }

proc, tissue = pathology_preprocess(train_df.iloc[0].image_path)
desc, aux = descriptor_map(proc, tissue)
print("Descriptor shape:", desc.shape)


In [ ]:

# ============================================================
# 9. PYTORCH DATASET
# ============================================================
class OsteosarcomaDataset(Dataset):
    def __init__(self, frame, augment=False):
        self.frame = frame.reset_index(drop=True)
        self.augment = augment

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img, tissue = pathology_preprocess(row.image_path)

        if self.augment:
            if random.random() < 0.5:
                img = np.flip(img, axis=1).copy()
                tissue = np.flip(tissue, axis=1).copy()
            if random.random() < 0.5:
                img = np.flip(img, axis=0).copy()
                tissue = np.flip(tissue, axis=0).copy()
            if random.random() < 0.25:
                k = random.choice([1,2,3])
                img = np.rot90(img, k).copy()
                tissue = np.rot90(tissue, k).copy()

        desc, aux = descriptor_map(img, tissue, grid=IMG_SIZE//PATCH_SIZE)

        x = torch.from_numpy(img.transpose(2,0,1)).float()
        d = torch.from_numpy(desc).float()
        tissue_t = torch.from_numpy(tissue[None]).float()

        return {
            "image": x,
            "descriptors": d,
            "tissue": tissue_t,
            "label": torch.tensor(int(row.label_id), dtype=torch.long),
            "path": row.image_path,
            "patient_id": str(row.patient_id)
        }

train_ds = OsteosarcomaDataset(train_df, augment=True)
val_ds = OsteosarcomaDataset(val_df, augment=False)
test_ds = OsteosarcomaDataset(test_df, augment=False)

# Weighted sampling for class imbalance.
counts = np.bincount(train_df.label_id.values, minlength=3)
sample_weights = train_df.label_id.map({i: 1/max(c,1) for i,c in enumerate(counts)}).values
sampler = WeightedRandomSampler(
    torch.as_tensor(sample_weights, dtype=torch.double),
    num_samples=len(sample_weights),
    replacement=True
)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=NUM_WORKERS, pin_memory=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=NUM_WORKERS, pin_memory=True)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=NUM_WORKERS, pin_memory=True)

print(len(train_ds), len(val_ds), len(test_ds))


## 10. PTRN — Pathology-aware relational representation

In [ ]:

# ============================================================
# 10. PTRN
# ============================================================
class PatchEmbed(nn.Module):
    def __init__(self, img_size=224, patch_size=16, embed_dim=128, desc_dim=10):
        super().__init__()
        self.proj = nn.Conv2d(3, embed_dim, patch_size, patch_size)
        self.pos = nn.Linear(2, embed_dim)
        self.desc = nn.Sequential(
            nn.Linear(desc_dim, 64), nn.GELU(), nn.Linear(64, embed_dim)
        )
        self.norm = nn.LayerNorm(embed_dim)

    def forward(self, x, desc):
        z = self.proj(x)                           # B,C,H',W'
        B,C,H,W = z.shape
        tok = z.flatten(2).transpose(1,2)          # B,N,C

        yy, xx = torch.meshgrid(
            torch.linspace(0,1,H,device=x.device),
            torch.linspace(0,1,W,device=x.device),
            indexing="ij"
        )
        coords = torch.stack([xx,yy], dim=-1).reshape(1,H*W,2)
        tok = tok + self.pos(coords)

        tok = tok + self.desc(desc)
        return self.norm(tok), (H,W)

class MultiScalePathology(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dw3 = nn.Conv1d(dim, dim, 3, padding=1, groups=dim)
        self.dw5 = nn.Conv1d(dim, dim, 5, padding=2, groups=dim)
        self.dw7 = nn.Conv1d(dim, dim, 7, padding=3, groups=dim)
        self.weight = nn.Sequential(
            nn.Linear(dim, dim//2), nn.GELU(), nn.Linear(dim//2, 3)
        )
        self.proj = nn.Linear(dim, dim)

    def forward(self, x):
        # x: B,N,D
        t = x.transpose(1,2)
        a = self.dw3(t).transpose(1,2)
        b = self.dw5(t).transpose(1,2)
        c = self.dw7(t).transpose(1,2)
        s = torch.softmax(self.weight(x), dim=-1)
        out = a*s[...,0:1] + b*s[...,1:2] + c*s[...,2:3]
        return self.proj(out)

class RelationalLayer(nn.Module):
    def __init__(self, dim=128, heads=4):
        super().__init__()
        self.heads = heads
        self.dh = dim // heads
        self.q = nn.Linear(dim, dim)
        self.k = nn.Linear(dim, dim)
        self.v = nn.Linear(dim, dim)
        self.out = nn.Linear(dim, dim)
        self.norm = nn.LayerNorm(dim)
        self.rel = nn.Sequential(
            nn.Linear(1, heads), nn.Tanh()
        )

    def forward(self, x, desc, top_k=8):
        B,N,D = x.shape
        h = self.heads
        q = self.q(x).view(B,N,h,self.dh).transpose(1,2)
        k = self.k(x).view(B,N,h,self.dh).transpose(1,2)
        v = self.v(x).view(B,N,h,self.dh).transpose(1,2)

        # Pathology descriptor distance drives neighborhood selection.
        dd = torch.cdist(desc, desc, p=1) / max(desc.shape[-1],1)
        kk = min(top_k, max(N-1,1))
        _, idx = torch.topk(-dd, k=kk, dim=-1)

        idx_k = idx.unsqueeze(1).expand(B,h,N,kk)
        k_expand = k.unsqueeze(2).expand(B,h,N,N,self.dh)
        v_expand = v.unsqueeze(2).expand(B,h,N,N,self.dh)
        k_nei = torch.gather(
            k_expand, 3, idx_k.unsqueeze(-1).expand(B,h,N,kk,self.dh)
        )
        v_nei = torch.gather(
            v_expand, 3, idx_k.unsqueeze(-1).expand(B,h,N,kk,self.dh)
        )

        q_ref = q.unsqueeze(3)
        logits = (q_ref * k_nei).sum(-1) / math.sqrt(self.dh)

        pair_d = torch.gather(
            dd, 2, idx
        ).unsqueeze(1).expand(B,h,N,kk)
        logits = logits + self.rel(pair_d).permute(0,3,2,1)
        attn = torch.softmax(logits, dim=-1)
        out = (attn.unsqueeze(-1) * v_nei).sum(3)
        out = out.transpose(1,2).reshape(B,N,D)
        return self.norm(x + self.out(out))

class PTRN(nn.Module):
    def __init__(self, img_size=224, patch=16, dim=128, desc_dim=10):
        super().__init__()
        self.embed = PatchEmbed(img_size, patch, dim, desc_dim)
        self.ms = MultiScalePathology(dim)
        self.layers = nn.ModuleList([
            RelationalLayer(dim, REL_HEADS) for _ in range(REL_LAYERS)
        ])
        self.pool = nn.Sequential(
            nn.Linear(dim, 1)
        )

    def forward(self, image, descriptors):
        tok, hw = self.embed(image, descriptors)
        tok = self.ms(tok)
        for layer in self.layers:
            tok = layer(tok, descriptors, TOP_K)

        a = torch.softmax(self.pool(tok).squeeze(-1), dim=-1)
        global_feat = (tok * a.unsqueeze(-1)).sum(1)
        spatial = tok.transpose(1,2).reshape(
            image.shape[0], tok.shape[-1], hw[0], hw[1]
        )
        return {"tokens": tok, "spatial": spatial, "global": global_feat, "attention": a}


## 11. TBEN — weakly supervised tumor boundary evolution

In [ ]:

# ============================================================
# 11. TBEN
# ============================================================
class CAMInitializer(nn.Module):
    def __init__(self, in_ch, num_classes=3):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, 64, 3, padding=1)
        self.head = nn.Conv2d(64, num_classes, 1)

    def forward(self, x):
        z = F.gelu(self.conv(x))
        logits = self.head(z)
        pooled = F.adaptive_avg_pool2d(logits,1).flatten(1)
        return logits, pooled

def signed_distance(mask):
    m = mask > 0.5
    inside = distance_transform_edt(m)
    outside = distance_transform_edt(~m)
    return (inside - outside).astype(np.float32)

class TBEN(nn.Module):
    def __init__(self, in_ch=128, steps=3):
        super().__init__()
        self.steps = steps
        self.decoder = nn.Sequential(
            nn.Conv2d(in_ch, 96, 3, padding=1),
            nn.BatchNorm2d(96), nn.GELU(),
            nn.Conv2d(96, 64, 3, padding=1),
            nn.BatchNorm2d(64), nn.GELU()
        )
        self.rel_head = nn.Conv2d(64, 1, 1)
        self.velocity = nn.Sequential(
            nn.Conv2d(64 + 4, 64, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(64, 1, 1),
            nn.Tanh()
        )
        self.reliability = nn.Sequential(
            nn.Conv2d(64 + 1, 32, 3, padding=1),
            nn.GELU(),
            nn.Conv2d(32,1,1),
            nn.Sigmoid()
        )

    def forward(self, spatial, tissue):
        feat = self.decoder(spatial)
        feat = F.interpolate(feat, size=tissue.shape[-2:], mode="bilinear", align_corners=False)

        # Weak initial tumor region: non-background tissue.
        init = tissue.clamp(0,1)

        # Smooth signed-distance-like state from the weak region.
        with torch.no_grad():
            sd_np = []
            for m in init[:,0].detach().cpu().numpy():
                sd_np.append(signed_distance(m))
            sd = torch.from_numpy(np.stack(sd_np)[:,None]).float().to(tissue.device)
        sd = sd / (sd.abs().amax(dim=(-2,-1), keepdim=True) + 1e-6)
        prob = torch.sigmoid(-sd * 4.0)

        states = [prob]
        for _ in range(self.steps):
            boundary = torch.exp(-torch.abs(sd) * 5.0)
            grad_y = F.pad(sd[:,:,1:]-sd[:,:,:-1], (0,0,0,1))
            grad_x = F.pad(sd[:,:,:,1:]-sd[:,:,:,:-1], (0,1,0,0))
            grad_mag = torch.sqrt(grad_x**2 + grad_y**2 + 1e-6)
            rel = self.reliability(torch.cat([feat, prob], dim=1))
            edge = grad_mag
            inp = torch.cat([feat, prob, boundary, rel, edge], dim=1)
            vel = self.velocity(inp) * TBEN_STEP_SIZE

            hard_band = (torch.abs(sd) < TBEN_BAND).float().detach()
            sd = sd + vel * hard_band
            prob = torch.sigmoid(-sd * 4.0)
            states.append(prob)

        return {
            "prob": prob,
            "states": states,
            "boundary": torch.exp(-torch.abs(sd) * 5.0),
            "reliability": rel,
            "feature": feat
        }

def pseudo_mask_from_tissue(tissue):
    # A robust fallback pseudo-mask when no pixel annotations are supplied.
    # Non-tumor images should ideally be handled by a trained CAM initializer;
    # this function is intentionally weak and can be replaced by CAM pseudo-labels.
    return tissue.clone()


## 12. PCCAECN / PARCEN — conflict-aware evidence consensus

In [ ]:

# ============================================================
# 12. PCCAECN / PARCEN
# ============================================================
class EvidenceStream(nn.Module):
    def __init__(self, in_dim, evidence_dim=64, classes=3):
        super().__init__()
        self.project = nn.Sequential(
            nn.Linear(in_dim, evidence_dim), nn.GELU(), nn.Dropout(0.1)
        )
        self.classifier = nn.Linear(evidence_dim, classes)

    def forward(self, x):
        z = self.project(x)
        logits = self.classifier(z)
        return z, logits

class PARCEN(nn.Module):
    def __init__(self, dims, evidence_dim=64, classes=3):
        super().__init__()
        self.streams = nn.ModuleList([
            EvidenceStream(d, evidence_dim, classes) for d in dims
        ])
        self.reliability = nn.Sequential(
            nn.Linear(evidence_dim + classes + 2, 64),
            nn.GELU(), nn.Linear(64, classes), nn.Sigmoid()
        )
        self.correction = nn.Sequential(
            nn.Linear(evidence_dim + classes + 2, 64),
            nn.GELU(), nn.Linear(64, classes)
        )
        self.temperature = nn.Parameter(torch.tensor(1.0))

    def forward(self, evidence, tumor_support, boundary_support):
        zs, logits, probs = [], [], []
        for stream, x in zip(self.streams, evidence):
            z, lg = stream(x)
            zs.append(z)
            logits.append(lg)
            probs.append(torch.softmax(lg, dim=-1))

        P = torch.stack(probs, dim=1)  # B,S,C
        mean_p = P.mean(dim=1, keepdim=True)
        conflict = torch.abs(P - mean_p).mean(dim=-1, keepdim=True)

        fused = []
        rels = []
        corrections = []
        for i,z in enumerate(zs):
            inp = torch.cat([
                z, probs[i], tumor_support, boundary_support
            ], dim=-1)
            r = self.reliability(inp)
            corr = self.correction(inp)
            corrected = torch.softmax(logits[i] + corr, dim=-1)
            rels.append(r)
            corrections.append(corr)
            fused.append(corrected * r)

        num = torch.stack(fused, dim=1).sum(dim=1)
        den = torch.stack(rels, dim=1).sum(dim=1) + 1e-6
        consensus = num / den
        final_logits = torch.log(consensus.clamp_min(1e-7)) / self.temperature.clamp_min(0.05)
        final_prob = torch.softmax(final_logits, dim=-1)

        return {
            "stream_logits": logits,
            "stream_probs": probs,
            "reliability": torch.stack(rels, dim=1),
            "conflict": conflict.squeeze(-1),
            "corrections": corrections,
            "logits": final_logits,
            "prob": final_prob
        }

class FullPTRNTBENPCCAECN(nn.Module):
    def __init__(self):
        super().__init__()
        self.ptrn = PTRN(IMG_SIZE, PATCH_SIZE, EMBED_DIM, 10)
        self.tben = TBEN(EMBED_DIM, TBEN_STEPS)

        # Five evidence streams from the manuscript:
        # semantic, tumor-region, boundary, texture, morphology.
        self.semantic_proj = nn.Linear(EMBED_DIM, 128)
        self.region_proj = nn.Linear(2, 32)
        self.boundary_proj = nn.Linear(2, 32)
        self.texture_proj = nn.Linear(10, 32)
        self.morph_proj = nn.Linear(10, 32)

        self.parcen = PARCEN([128,32,32,32,32], 64, 3)

    def forward(self, image, descriptors, tissue):
        ptr = self.ptrn(image, descriptors)
        tb = self.tben(ptr["spatial"], tissue)

        sem = self.semantic_proj(ptr["global"])
        tumor_support = tb["prob"].mean(dim=(2,3), keepdim=False).unsqueeze(-1)
        boundary_support = tb["boundary"].mean(dim=(2,3), keepdim=False).unsqueeze(-1)

        region = torch.cat([
            tumor_support,
            (tb["prob"] * tissue).mean(dim=(2,3)).unsqueeze(-1)
        ], dim=-1)
        boundary = torch.cat([
            boundary_support,
            (tb["boundary"] * tissue).mean(dim=(2,3)).unsqueeze(-1)
        ], dim=-1)

        tex = descriptors.mean(dim=1)
        # Split the 10 descriptors into texture and morphology summaries.
        texture = self.texture_proj(tex)
        morph = self.morph_proj(tex)

        ev = [sem, self.region_proj(region), self.boundary_proj(boundary), texture, morph]
        out = self.parcen(ev, tumor_support, boundary_support)

        out["ptrn"] = ptr
        out["tben"] = tb
        return out


In [ ]:

# ============================================================
# 13. LOSSES
# ============================================================
def dice_loss(pred, target, eps=1e-6):
    pred = pred.flatten(1)
    target = target.flatten(1)
    inter = (pred * target).sum(1)
    return (1 - (2*inter + eps)/(pred.sum(1)+target.sum(1)+eps)).mean()

def focal_ce(logits, target, gamma=2.0, alpha=None):
    ce = F.cross_entropy(logits, target, weight=alpha, reduction="none")
    pt = torch.exp(-ce)
    return ((1-pt)**gamma * ce).mean()

def total_loss(out, label, tissue):
    cls = focal_ce(out["logits"], label)

    # Weak localization objective: tissue-supported tumor region.
    # For non-tumor, target is zero; for tumor classes, tissue is a weak target.
    loc_target = tissue * (label[:,None,None,None] != 0).float()
    loc = 0.5 * dice_loss(out["tben"]["prob"], loc_target) +           0.5 * F.binary_cross_entropy(out["tben"]["prob"], loc_target)

    # Deep supervision over intermediate TBEN states.
    deep = 0.0
    for s in out["tben"]["states"]:
        deep = deep + 0.25 * F.binary_cross_entropy(s, loc_target)

    # Auxiliary stream classification.
    aux = 0.0
    for lg in out["stream_logits"]:
        aux = aux + 0.15 * F.cross_entropy(lg, label)

    return cls + 0.4*loc + deep + aux


In [ ]:

# ============================================================
# 14. MODEL + OPTIMIZER
# ============================================================
model = FullPTRNTBENPCCAECN().to(DEVICE)

optimizer = torch.optim.AdamW(
    model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS
)

print("Trainable parameters:",
      sum(p.numel() for p in model.parameters() if p.requires_grad)/1e6, "M")


In [ ]:

# ============================================================
# 15. TRAIN / EVALUATE FUNCTIONS
# ============================================================
def move_batch(batch):
    return (
        batch["image"].to(DEVICE, non_blocking=True),
        batch["descriptors"].to(DEVICE, non_blocking=True),
        batch["tissue"].to(DEVICE, non_blocking=True),
        batch["label"].to(DEVICE, non_blocking=True)
    )

def run_epoch(loader, train=True):
    model.train(train)
    total = 0.0
    ys, ps, probs = [], [], []

    for batch in tqdm(loader, leave=False):
        image, desc, tissue, y = move_batch(batch)

        with torch.set_grad_enabled(train):
            out = model(image, desc, tissue)
            loss = total_loss(out, y, tissue)

            if train:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
                optimizer.step()

        total += float(loss.detach().cpu()) * len(y)
        ys.extend(y.detach().cpu().numpy().tolist())
        ps.extend(out["prob"].argmax(1).detach().cpu().numpy().tolist())
        probs.extend(out["prob"].detach().cpu().numpy().tolist())

    acc = accuracy_score(ys, ps)
    pr, rc, f1, _ = precision_recall_fscore_support(
        ys, ps, average="macro", zero_division=0
    )
    mcc = matthews_corrcoef(ys, ps)
    return total/len(loader.dataset), acc, pr, rc, f1, mcc, np.asarray(ys), np.asarray(ps), np.asarray(probs)

history = []
best_val = float("inf")
bad_epochs = 0
best_state = None

for epoch in range(EPOCHS):
    t0 = time.time()
    tr = run_epoch(train_loader, train=True)
    va = run_epoch(val_loader, train=False)
    scheduler.step()

    history.append({
        "epoch": epoch+1,
        "train_loss": tr[0], "train_acc": tr[1],
        "val_loss": va[0], "val_acc": va[1],
        "val_precision": va[2], "val_recall": va[3],
        "val_f1": va[4], "val_mcc": va[5]
    })

    print(
        f"Epoch {epoch+1:02d}/{EPOCHS} | "
        f"Train loss {tr[0]:.4f} acc {tr[1]:.4f} | "
        f"Val loss {va[0]:.4f} acc {va[1]:.4f} f1 {va[4]:.4f} | "
        f"{time.time()-t0:.1f}s"
    )

    if va[0] < best_val:
        best_val = va[0]
        bad_epochs = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, os.path.join(SAVE_DIR, "best_PTRN_TBEN_PCCAECN.pth"))
    else:
        bad_epochs += 1
        if bad_epochs >= PATIENCE:
            print("Early stopping.")
            break

if best_state is not None:
    model.load_state_dict(best_state)

hist_df = pd.DataFrame(history)
hist_df.to_csv(os.path.join(SAVE_DIR, "training_history.csv"), index=False)


In [ ]:

# ============================================================
# 16. TEST METRICS
# ============================================================
test_out = run_epoch(test_loader, train=False)
_, acc, precision, recall, f1, mcc, y_true, y_pred, y_prob = test_out

print(classification_report(
    y_true, y_pred, target_names=CLASS_NAMES, digits=4, zero_division=0
))
print("Accuracy :", round(acc, 4))
print("Precision:", round(precision, 4))
print("Recall   :", round(recall, 4))
print("F1       :", round(f1, 4))
print("MCC      :", round(mcc, 4))

try:
    auc = roc_auc_score(y_true, y_prob, multi_class="ovr", average="macro")
except Exception:
    auc = np.nan
print("Macro ROC-AUC:", auc)

metrics_df = pd.DataFrame({
    "Metric": ["Accuracy","Precision","Recall","F1-score","MCC","Macro ROC-AUC"],
    "Value": [acc, precision, recall, f1, mcc, auc]
})
metrics_df.to_excel(os.path.join(SAVE_DIR, "test_metrics.xlsx"), index=False)
display(metrics_df)


In [ ]:

# ============================================================
# 17. CONFUSION MATRIX
# ============================================================
cm = confusion_matrix(y_true, y_pred, labels=[0,1,2])

plt.figure(figsize=(7,6), dpi=180)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
            annot_kws={"size":14, "weight":"bold"})
plt.xlabel("Predicted", fontsize=14, fontweight="bold")
plt.ylabel("True", fontsize=14, fontweight="bold")
plt.title("PTRN–TBEN–PCCAECN Confusion Matrix", fontsize=15, fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 18. TRAINING ACCURACY / LOSS CURVES
# ============================================================
fig, ax = plt.subplots(1,2, figsize=(14,5), dpi=160)

ax[0].plot(hist_df.epoch, hist_df.train_acc, marker="o", label="Train")
ax[0].plot(hist_df.epoch, hist_df.val_acc, marker="s", label="Validation")
ax[0].set_title("Training and Validation Accuracy", fontweight="bold")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("Accuracy")
ax[0].legend()

ax[1].plot(hist_df.epoch, hist_df.train_loss, marker="o", label="Train")
ax[1].plot(hist_df.epoch, hist_df.val_loss, marker="s", label="Validation")
ax[1].set_title("Training and Validation Loss", fontweight="bold")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Loss")
ax[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "training_curves.png"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 19. LOCALIZATION VISUALIZATION
# ============================================================
@torch.no_grad()
def predict_one(path):
    temp = pd.DataFrame([{
        "image_path": path,
        "label": "Non-Tumor",
        "label_id": 0,
        "patient_id": infer_patient_id(path)
    }])
    ds = OsteosarcomaDataset(temp, augment=False)
    item = ds[0]

    image = item["image"][None].to(DEVICE)
    desc = item["descriptors"][None].to(DEVICE)
    tissue = item["tissue"][None].to(DEVICE)

    out = model(image, desc, tissue)
    return item, out

sample_path = test_df.iloc[0].image_path
item, out = predict_one(sample_path)

raw = np.array(Image.open(sample_path).convert("RGB").resize((IMG_SIZE, IMG_SIZE)))
prob = out["tben"]["prob"][0,0].cpu().numpy()
boundary = out["tben"]["boundary"][0,0].cpu().numpy()
pred_class = int(out["prob"].argmax(1).item())
confidence = float(out["prob"].max().item())

fig, ax = plt.subplots(1,4, figsize=(18,5), dpi=180)
ax[0].imshow(raw); ax[0].set_title("Input", fontweight="bold")
ax[1].imshow(prob, cmap="magma"); ax[1].set_title("TBEN tumor probability", fontweight="bold")
ax[2].imshow(raw); ax[2].imshow(prob, cmap="jet", alpha=0.40)
ax[2].set_title("Localization overlay", fontweight="bold")
ax[3].imshow(raw)
ax[3].contour(prob > 0.5, levels=[0.5], linewidths=2)
ax[3].set_title(f"{CLASS_NAMES[pred_class]} ({confidence:.3f})", fontweight="bold")
for a in ax: a.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "TBEN_localization.png"), dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 20. INFERENCE-TIME MORPHOLOGICAL CONSISTENCY FILTER
# ============================================================
def morphological_consistency(prob, threshold=0.5, min_area=50):
    mask = prob >= threshold
    mask = remove_small_objects(mask, min_size=min_area)
    mask = remove_small_holes(mask, area_threshold=min_area)
    return mask.astype(np.uint8)

filtered = morphological_consistency(prob)

plt.figure(figsize=(6,5), dpi=180)
plt.imshow(filtered, cmap="gray")
plt.title("TBEN inference-time filtered localization", fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "TBEN_filtered_mask.png"), dpi=600, bbox_inches="tight")
plt.show()



## 21. Evidence-stream analysis

The five streams follow the manuscript's late evidence integration idea:

1. Semantic evidence
2. Tumor-region evidence
3. Boundary evidence
4. Texture evidence
5. Morphological evidence

Conflict is measured from disagreement with the consensus posterior, while reliability gates
the final class-specific evidence contribution.


In [ ]:

# ============================================================
# 21. EVIDENCE RELIABILITY / CONFLICT
# ============================================================
@torch.no_grad()
def evidence_report(path):
    item, out = predict_one(path)
    probs = [p[0].cpu().numpy() for p in out["parcen"]["stream_probs"]] if "parcen" in out else []
    # predict_one returns flattened output keys; reconstruct from current model output:
    # Full model returns stream probabilities under out.
    probs = [p[0].cpu().numpy() for p in out["stream_probs"]]
    rel = out["reliability"][0].cpu().numpy()
    conflict = out["conflict"][0].cpu().numpy()

    rows = []
    names = ["Semantic","Tumor-region","Boundary","Texture","Morphology"]
    for i,n in enumerate(names):
        rows.append({
            "Evidence stream": n,
            "Reliability mean": float(rel[i].mean()),
            "Conflict": float(conflict[i]),
            "Predicted class": CLASS_NAMES[int(np.argmax(probs[i]))],
            "Max probability": float(np.max(probs[i]))
        })
    return pd.DataFrame(rows)

ev_df = evidence_report(sample_path)
display(ev_df)
ev_df.to_excel(os.path.join(SAVE_DIR, "evidence_stream_analysis.xlsx"), index=False)


In [ ]:

# ============================================================
# 22. INTEGRATED-GRADIENTS-STYLE IMAGE ATTRIBUTION
# ============================================================
# Practical implementation over the final classification logit.
# The paper specifies a fixed training-set mean RGB baseline.

def integrated_gradients(path, steps=24):
    item, out0 = predict_one(path)
    image = item["image"][None].to(DEVICE)
    desc = item["descriptors"][None].to(DEVICE)
    tissue = item["tissue"][None].to(DEVICE)

    baseline = torch.full_like(image, float(image.mean().detach().cpu()))
    pred_class = int(out0["prob"].argmax(1).item())

    total_grad = torch.zeros_like(image)

    for a in torch.linspace(0,1,steps,device=DEVICE):
        x = (baseline + a*(image-baseline)).detach().requires_grad_(True)
        out = model(x, desc, tissue)
        score = out["logits"][0,pred_class]
        model.zero_grad(set_to_none=True)
        score.backward()
        total_grad += x.grad.detach()

    attr = (image-baseline) * total_grad / steps
    attr = attr.abs().sum(1)[0]
    attr = attr / (attr.max()+1e-8)
    return attr.cpu().numpy(), pred_class

attr, pc = integrated_gradients(sample_path)

plt.figure(figsize=(7,6), dpi=180)
plt.imshow(raw)
plt.imshow(attr, cmap="inferno", alpha=0.45)
plt.title(f"Integrated-gradient attribution: {CLASS_NAMES[pc]}", fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "integrated_gradients_attribution.png"),
            dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 23. SIMPLE LIME-STYLE SUPERPIXEL ABLATION MAP
# ============================================================
# This is a model-agnostic local perturbation explanation.
from skimage.segmentation import slic

@torch.no_grad()
def predict_probability_from_tensor(x, desc, tissue):
    return model(x, desc, tissue)["prob"][0].cpu().numpy()

def lime_style_explanation(path, n_segments=50):
    item, out = predict_one(path)
    image = item["image"][None].to(DEVICE)
    desc = item["descriptors"][None].to(DEVICE)
    tissue = item["tissue"][None].to(DEVICE)

    img_np = item["image"].permute(1,2,0).numpy()
    img_np = np.clip(img_np,0,1)
    segments = slic(img_np, n_segments=n_segments, compactness=8, start_label=0)

    base = out["prob"][0].detach().cpu().numpy()
    cls = int(base.argmax())
    scores = np.zeros(segments.max()+1, dtype=np.float32)

    for s in range(len(scores)):
        x = image.clone()
        m = torch.from_numpy((segments==s).astype(np.float32)).to(DEVICE)
        x = x * (1-m[None,None]) + x.mean(dim=(2,3), keepdim=True) * m[None,None]
        p = predict_probability_from_tensor(x, desc, tissue)[cls]
        scores[s] = base[cls] - p

    heat = scores[segments]
    heat = (heat-heat.min())/(heat.max()-heat.min()+1e-8)
    return img_np, heat, cls

lime_img, lime_heat, lime_cls = lime_style_explanation(sample_path)

plt.figure(figsize=(7,6), dpi=180)
plt.imshow(lime_img)
plt.imshow(lime_heat, cmap="viridis", alpha=0.45)
plt.title(f"LIME-style local explanation: {CLASS_NAMES[lime_cls]}", fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "lime_style_explanation.png"),
            dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 24. CLASS DISTRIBUTION
# ============================================================
dist = df["label"].value_counts().reindex(CLASS_NAMES).fillna(0)

plt.figure(figsize=(8,5), dpi=180)
ax = dist.plot(kind="bar")
plt.title("Osteosarcoma Histopathology Class Distribution", fontweight="bold")
plt.xlabel("Class", fontweight="bold")
plt.ylabel("Number of images", fontweight="bold")
plt.xticks(rotation=0)
for p in ax.patches:
    ax.annotate(str(int(p.get_height())),
                (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom", fontweight="bold")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "data_distribution.png"),
            dpi=600, bbox_inches="tight")
plt.show()


In [ ]:

# ============================================================
# 25. OPTIONAL 5-FOLD PATIENT-LEVEL EVALUATION
# ============================================================
# This section evaluates a freshly initialized lightweight version of the same architecture
# across patient-level folds. For a full expensive 5-fold training, set RUN_5FOLD=True.
RUN_5FOLD = False

if RUN_5FOLD:
    sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
    fold_rows = []

    for fold, (tr_idx, te_idx) in enumerate(
        sgkf.split(df, df["label_id"], groups=df["patient_id"]), 1
    ):
        tr = df.iloc[tr_idx].reset_index(drop=True)
        te = df.iloc[te_idx].reset_index(drop=True)

        tr_loader = DataLoader(
            OsteosarcomaDataset(tr, augment=True),
            batch_size=BATCH_SIZE, shuffle=True,
            num_workers=NUM_WORKERS
        )
        te_loader = DataLoader(
            OsteosarcomaDataset(te, augment=False),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=NUM_WORKERS
        )

        fold_model = FullPTRNTBENPCCAECN().to(DEVICE)
        fold_opt = torch.optim.AdamW(
            fold_model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY
        )

        old_model, old_opt = model, optimizer
        model, optimizer = fold_model, fold_opt

        for ep in range(max(2, EPOCHS//2)):
            run_epoch(tr_loader, train=True)

        res = run_epoch(te_loader, train=False)
        fold_rows.append({
            "Fold": fold,
            "Accuracy": res[1],
            "Precision": res[2],
            "Recall": res[3],
            "F1": res[4],
            "MCC": res[5]
        })

        model, optimizer = old_model, old_opt
        del fold_model, fold_opt
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    cv_df = pd.DataFrame(fold_rows)
    cv_df.loc["Mean"] = cv_df.mean(numeric_only=True)
    cv_df.loc["Std"] = cv_df.std(numeric_only=True)
    display(cv_df)
    cv_df.to_excel(os.path.join(SAVE_DIR, "five_fold_patient_level_results.xlsx"), index=False)
else:
    print("RUN_5FOLD=False. Enable it for full patient-level 5-fold training.")


In [ ]:

# ============================================================
# 26. SAVE COMPLETE RESULTS PACKAGE
# ============================================================
import shutil

results = {
    "test_metrics": metrics_df,
    "training_history": hist_df,
    "evidence_stream_analysis": ev_df
}

xlsx_path = os.path.join(SAVE_DIR, "PTRN_TBEN_PCCAECN_results.xlsx")
with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    for name, table in results.items():
        table.to_excel(writer, sheet_name=name[:31], index=False)

# Save test predictions.
pred_df = test_df.copy()
pred_df["predicted_label"] = [CLASS_NAMES[i] for i in y_pred]
pred_df["confidence"] = y_prob.max(axis=1)
for i,c in enumerate(CLASS_NAMES):
    pred_df[f"prob_{c}"] = y_prob[:,i]
pred_df.to_csv(os.path.join(SAVE_DIR, "test_predictions.csv"), index=False)

zip_path = shutil.make_archive(
    os.path.join(DRIVE_ROOT, "PTRN_TBEN_PCCAECN_Outputs"),
    "zip",
    SAVE_DIR
)

print("Saved:")
print(xlsx_path)
print(os.path.join(SAVE_DIR, "test_predictions.csv"))
print(zip_path)



## 27. Implementation notes

### What is directly represented
The notebook follows the supplied manuscript's stated processing and architectural sequence:
Macenko/Otsu/median/CLAHE/min-max preprocessing → pathology descriptors → PTRN relational
learning → TBEN weak localization and iterative boundary evolution → five-stream conflict-aware
evidence integration → calibrated prediction and attribution.

### Important reproducibility limitation
The manuscript describes a weakly supervised setting because complete pixel-level expert masks are
not available. The exact CAM inner-cross-fitting protocol and all numerical hyperparameters/results
are not fully specified in executable form in the document. This notebook therefore provides a
working implementation with explicit, replaceable defaults rather than pretending that unspecified
values are known.

### Recommended dataset layout
For best results, use:

```text
Dataset/
├── Non-Tumor/
├── Viable Tumor/
├── Necrotic Tumor/
└── manifest.csv   # optional
```

A manifest can contain:

```text
image_path,label,patient_id
Non-Tumor/img001.png,Non-Tumor,P001
Viable Tumor/img002.png,Viable Tumor,P002
Necrotic Tumor/img003.png,Necrotic Tumor,P003
```

### Clinical/research caution
This is research code and not a clinically validated diagnostic system. The notebook does not
create missing ground-truth annotations or guarantee the performance numbers in the manuscript.
